# Module 7 — A Scikit-learn Transformer

**By the end of this notebook, you will be able to:**
- Explain why a plain cleaning function cannot guarantee "learn from train, apply to anything" the way `Pipeline`, `GridSearchCV` and `cross_val_score` need it to
- Write a scikit-learn-compatible transformer (`BaseEstimator`, `TransformerMixin`) whose `fit` learns only from the data it is given, and whose `transform` applies that frozen decision to any data it is given afterward
- Plug a transformer into a real `sklearn.pipeline.Pipeline` and evaluate it with `cross_val_score` — no more hand-written per-fold loops

**Context:** Since Module 2, `columns_above_missing_threshold` has always been computed once, on the whole 4-city dataset, before `evaluate_group_cv` ever splits into folds — which means a "held-out" fold's own city can quietly influence which columns get dropped for it. At Module 2's threshold (0.7) this happens not to change anything on this dataset; at a more aggressive threshold like 0.6 it does — verified end-to-end, this raises the reported rmse_mean from an optimistic 28.14 (today's approach) to a more honest 31.05. If you want to see it for yourself: recompute `columns_above_missing_threshold` at threshold 0.6 on all four cities, then again after excluding one city at a time, and compare the two lists of dropped columns. This notebook does not chase that number further — it teaches the scikit-learn tool that fixes the *process* itself, so this kind of leak becomes structurally impossible regardless of threshold.

## Why scikit-learn has a `fit`/`transform` contract

Any real pipeline needs preparation steps that *learn* something from the training data — a mean, a set of columns to drop, a scaling factor — and then *apply* that exact learned decision to other data later, without ever recomputing it. A plain function cannot make that promise: nothing stops you from accidentally calling it again on the wrong data, and — more importantly for this course — tools like [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html), [`GridSearchCV`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.GridSearchCV.html) and `cross_val_score` cannot automate "fit only on this fold's training rows, then apply to its validation rows" unless every step in the chain follows the *same* interface.

That interface is two methods and two base classes:
- [`BaseEstimator`](https://scikit-learn.org/stable/modules/generated/sklearn.base.BaseEstimator.html) gives you `get_params()`/`set_params()` for free, which is how `clone()` (used internally by every fold of `cross_val_score`/`GridSearchCV`) can produce a fresh, unfitted copy of your transformer between folds
- [`TransformerMixin`](https://scikit-learn.org/stable/modules/generated/sklearn.base.TransformerMixin.html) gives you `fit_transform(X)` for free, implemented as `fit(X).transform(X)`

The full rulebook is in scikit-learn's own [Developing scikit-learn estimators](https://scikit-learn.org/stable/developers/develop.html) guide. The two rules that matter most here:
- `fit(X, y=None)` learns whatever it needs **from `X` alone**, stores it in attributes ending with `_` (the scikit-learn convention for "set during fit"), and returns `self`
- `transform(X)` applies what was learned to **whatever `X` it is given**, train, validation, or genuinely new data — it never recomputes anything from that `X`

In [ ]:
import numpy as np
import pandas as pd

train_toy = pd.DataFrame({"x1": [1.0, 2.0, np.nan, 4.0], "x2": [10.0, np.nan, 30.0, 40.0]})
test_toy = pd.DataFrame({"x1": [np.nan, 5.0], "x2": [100.0, np.nan]})
train_toy

## A plain function cannot keep the promise

Here is the naive way to fill missing values with a column's mean:

```python
def fill_with_mean(df):
    return df.fillna(df.mean())
```

Call it on `train_toy`, then on `test_toy` — the same function, applied the same way.

In [ ]:
def fill_with_mean(df):
    return df.fillna(df.mean())


print("naive on train_toy:")
print(fill_with_mean(train_toy))
print("\nnaive on test_toy:")
print(fill_with_mean(test_toy))

**Look at `test_toy`'s result.** `x1` has one real value (`5.0`) and one `NaN` — `df.mean()` on `test_toy` alone is `5.0`, so the function fills the missing `x1` with `5.0`: it is filling a missing value with *itself*, a circular, meaningless "decision" that only happens because the function recomputes its statistic on whatever it is handed. If `test_toy` represents new data your model will see in production, this is not a training-time convenience going slightly stale — it is silently wrong every single time it runs, because the function was never designed to remember anything from training in the first place. A transformer fixes this structurally: `fit` learns the mean once, from training data only; `transform` reuses that exact number forever after.

## Freeze the decision with a class: `__init__`

`__init__` stores configuration only — every argument, unchanged, as an attribute of the exact same name. Nothing is computed here; that is `fit`'s job. This is not a style preference: `BaseEstimator.get_params()` inspects `__init__`'s signature and reads back attributes of the same name, and `clone()` (which `cross_val_score`/`GridSearchCV` call between every fold) rebuilds a fresh instance from exactly those params. Breaking this rule — computing something in `__init__`, or storing an argument under a different name — makes `clone()` silently produce a different object than you intended.

Our first transformer takes one optional parameter: which columns to impute (`None` means "all of them").

In [ ]:
from sklearn.base import BaseEstimator, TransformerMixin


class ColumnMeanImputer(BaseEstimator, TransformerMixin):
    """Fill missing values with each column's mean, learned once from fit's X."""

    def __init__(self, columns: list[str] | None = None):
        self.columns = columns


ColumnMeanImputer().get_params()

## `fit`: learn the means, and only the means

`fit(X, y=None)` receives `X` (and `y`, unused here — it exists only so every transformer accepts the same call signature as an estimator) and must compute the mean of each target column **using only this `X`**. Store the result in `self.means_` (trailing underscore: "set by fit") and return `self`, so `fit` can be chained as `fit(X).transform(X)`.

In [ ]:
# TODO: implement fit(X, y=None): compute the mean of each column in
# self.columns (or every column of X if self.columns is None), store the
# result in self.means_, and return self
def fit(self, X, y=None):
    cols = self.columns if self.columns is not None else list(X.columns)
    self.means_ = X[cols].mean()
    return self


ColumnMeanImputer.fit = fit

imputer = ColumnMeanImputer()
imputer.fit(train_toy)
imputer.means_

## `transform`: apply what was learned, nothing else

`transform(X)` must use `self.means_` — set once, by `fit`, on `train_toy` — to fill missing values in *whatever* `X` it receives now. It never looks at `X`'s own mean.

In [ ]:
# TODO: implement transform(X): fill missing values in the learned columns
# using self.means_ (never recompute a mean from the X given here); return
# the filled DataFrame
def transform(self, X):
    X = X.copy()
    cols = list(self.means_.index)
    X[cols] = X[cols].fillna(self.means_)
    return X


ColumnMeanImputer.transform = transform

imputer.transform(test_toy)

**Compare to the naive function's result on `test_toy` above.** `x1`'s missing value is now filled with `2.33` — `train_toy`'s mean, learned once and reused — instead of `5.0`, `test_toy`'s own circular mean. Same missing value, same row; the only thing that changed is *where* the fill number came from. That is the entire point of the contract.

## Free extras: `fit_transform` and `clone`

`TransformerMixin` already gave you `fit_transform(X)` — it just calls `fit(X).transform(X)`, nothing more. And `BaseEstimator` gave you compatibility with [`clone()`](https://scikit-learn.org/stable/modules/generated/sklearn.base.clone.html): given a transformer, `clone()` returns a **new, unfitted** instance built from `get_params()` — the exact mechanism `cross_val_score` and `GridSearchCV` use internally to get a fresh, blank copy for every single fold, so nothing learned on one fold can ever leak into another.

In [ ]:
from sklearn.base import clone

# fit_transform == fit(X).transform(X)
print("fit_transform matches fit().transform():",
      ColumnMeanImputer().fit_transform(train_toy).equals(imputer.transform(train_toy)))

# clone gives a fresh, unfitted copy — same config, no learned state
fresh = clone(imputer)
print("clone keeps params:", fresh.get_params())
print("clone has learned means_:", hasattr(fresh, "means_"))

## Compose it into a real `Pipeline`

Now the payoff: plug `ColumnMeanImputer` into a [`Pipeline`](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html) with a model, and evaluate with `cross_val_score` + `GroupKFold` — no hand-written per-fold loop, the way every module until now has done it. `Pipeline` and `cross_val_score` call `fit` **only on each fold's training rows**, then `transform` on both that fold's training and validation rows — automatically, for every fold, because both `ColumnMeanImputer` and the model share the same `fit`/`transform`/`clone` contract.

A synthetic grouped dataset, so `GroupKFold` has groups to hold out — no `air_quality` data needed yet.

In [ ]:
from sklearn.datasets import make_regression

X_arr, y_arr = make_regression(n_samples=40, n_features=3, noise=5.0, random_state=42)
toy_df = pd.DataFrame(X_arr, columns=["f1", "f2", "f3"])
toy_df.loc[[2, 10, 25], "f1"] = np.nan
toy_df.loc[[5, 30], "f2"] = np.nan
groups = np.repeat(["group_a", "group_b", "group_c", "group_d"], 10)
toy_df.isna().sum()

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GroupKFold, cross_val_score
from sklearn.pipeline import Pipeline

pipe = Pipeline([("imputer", ColumnMeanImputer()), ("model", LinearRegression())])
scores = cross_val_score(
    pipe, toy_df, y_arr, groups=groups, cv=GroupKFold(n_splits=4),
    scoring="neg_root_mean_squared_error",
)
print("rmse per fold:", -scores)
print("rmse mean:", -scores.mean())

**What just happened, fold by fold:** `cross_val_score` clones `pipe` fresh for every fold, calls `pipe.fit(train_rows, y_train)` — which itself calls `imputer.fit(train_rows)` (learning that fold's own means) then `model.fit(imputer.transform(train_rows), y_train)` — and finally scores it by calling `imputer.transform(validation_rows)` (using the means learned from *training* rows only) before `model.predict(...)`. The held-out group's own values never influence which mean fills its own missing data. This is exactly the guarantee Modules 2-5's `columns_above_missing_threshold` did not have.

## Reflection

**Question:** If `ColumnMeanImputer.fit` had instead computed its means from some dataset loaded from disk, or from a module-level global variable, instead of only from the `X` it receives — would `clone()` still protect you from leakage between folds? Why or why not? What does this tell you about *why* `columns_above_missing_threshold`, called once on the whole 4-city dataset before any fold split, has exactly the same structural problem as a badly written `fit`?

_Your answer here._

## Now do it for real: `air_quality`

Everything above was a toy example so you could see the mechanism clearly, with no `air_quality` code involved. The transfer to the real project is not a copy-paste — the learned decision is different (*which columns to drop*, by missingness threshold, not *column means*) — but the contract is identical.

**Exercise:** Open `src/air_quality/transformers.py` — `AirQualityCleaner(BaseEstimator, TransformerMixin)` is already scaffolded with the same shape as `ColumnMeanImputer` above. Its docstring and `tests/test_transformers.py` specify exactly what `fit` and `transform` must do:
- `fit(X, y=None)`: decide which columns to drop, using only `X` — reuse `data.columns_above_missing_threshold` exactly as Module 2 does, just on whatever `X` it is given instead of the whole dataset
- `transform(X)`: drop those columns from `X` with `data.drop_columns`, then fill what remains per city with `data.fill_missing_by_city`, then return only the numeric feature columns (`features.feature_columns`)

Run `uv run pytest tests/test_transformers.py -v` until it passes.

## Wire it into the pipeline

**Exercise:** Update `run_advanced` in `src/air_quality/workflows.py`. Replace the current "clean once on the whole dataset, then `evaluate_group_cv`" sequence with a real `Pipeline`:

```python
pipe = Pipeline([("cleaner", AirQualityCleaner(missing_threshold=0.6)), ("model", ...)])
scores = cross_val_score(pipe, enriched, enriched["pm2_5"], groups=enriched["city"], cv=GroupKFold(n_splits=4), scoring=...)
```

Use `missing_threshold=0.6`, not Module 2's `0.7` — at `0.7` the leak happens not to change the outcome on this dataset (see the warning on this module's page for why), so `0.6` is the threshold that actually demonstrates the fix. There is deliberately no fixed test for `run_advanced`'s exact shape, as with every module since Module 2 — verify it by running the cell below.

In [ ]:
# Once you have updated run_advanced in workflows.py, this confirms the fix
from air_quality.workflows import run_advanced

run_advanced()

## Wrap-up

Compare the rmse printed above to Module 2-5's number (~28). If you used `missing_threshold=0.6`, it should now land closer to the honest ~31.05 mentioned in this notebook's introduction, not the optimistic ~28.14 — the leak this module set out to fix. Write down, in one sentence, what `fit`/`transform` guarantee that calling a cleaning function directly, before any fold split, does not.

_Your observations here._